LLM-Generated Synthetic Data for Spurious Information Detection

Setup & imports

In [ ]:
import sys
import torch
import transformers

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "VRAM:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_NAME = "Qwen/Qwen3-8B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

llm = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

print("Model loaded successfully.")
print("Model:", MODEL_NAME)
print("Device:", llm.device)
print("GPU:", torch.cuda.get_device_name(0))

Data Loading

In [ ]:
import pandas as pd

ABSTRACTS_PATH = "Task 2/Task 2.1/source_abstracts.jsonl"

abstracts = pd.read_json(
    ABSTRACTS_PATH,
    lines=True
)

print("Abstracts shape:", abstracts.shape)
print("Columns:", abstracts.columns.tolist())

display(abstracts.head())

Synthetic Data Generation

In [ ]:
def extract_explicit_facts(abs_id, abstract):

    prompt = f"""
You are extracting factual information from a scientific abstract.

Your task is to identify ONLY facts that are explicitly stated in the
abstract.

Rules:

1. Every fact must be directly supported by the abstract.
2. Do not infer anything.
3. Do not add background knowledge.
4. Do not make comparisons unless the abstract explicitly makes them.
5. Do not interpret results.
6. Do not add conclusions.
7. Do not combine separate facts into a new claim.
8. Preserve numerical values exactly.
9. Preserve named methods, models, datasets and experimental details.
10. Each fact should be independently understandable.

Extract as many distinct factual claims as possible, ideally 10-20.

Return ONLY valid JSON.

Required format:

{{
  "facts": [
    "explicit fact 1",
    "explicit fact 2",
    "explicit fact 3"
  ]
}}

ABSTRACT:

{abstract}
"""

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=4096
    ).to(llm.device)

    with torch.no_grad():
        outputs = llm.generate(
            **inputs,
            max_new_tokens=800,
            do_sample=False
        )

    generated_text = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    try:
        result = json.loads(generated_text)
    except json.JSONDecodeError:
        print("JSON ERROR")
        print(generated_text)
        return []

    return result.get("facts", [])

In [ ]:
facts = extract_explicit_facts(
    abstracts.iloc[0]["abs_id"],
    abstracts.iloc[0]["abs_source"]
)

print("Number of facts:", len(facts))

for i, fact in enumerate(facts, 1):
    print(f"{i}. {fact}")

In [ ]:
def paraphrase_facts(abs_id, facts, num_sentences=10):

    facts_text = "\n".join(
        f"{i+1}. {fact}"
        for i, fact in enumerate(facts)
    )

    prompt = f"""
You are generating synthetic training examples for a scientific
text classification dataset.

The sentences you generate MUST be supported by the provided facts.

Below is a list of factual statements extracted from a scientific
abstract.

Your task is to generate {num_sentences} NEW sentences by paraphrasing
these facts.

STRICT RULES:

1. Every generated sentence MUST be supported by one or more of the
   provided facts.
2. Do not introduce information that is not present in the facts.
3. Do not use outside knowledge.
4. Do not add new methods, datasets, experiments, results, participants,
   locations, technologies, or conclusions.
5. Do not make new comparisons.
6. Do not strengthen or weaken claims.
7. Preserve numerical values exactly.
8. Preserve named models and methods accurately.
9. Do not change the meaning of the facts.
10. Do not copy the facts word-for-word.
11. Each sentence must be a natural scientific statement.
12. Prefer using different facts so the generated sentences are diverse.

IMPORTANT:

If a fact says that a model "compares favorably" with existing methods,
do not turn that into "the model is superior to all existing methods."

If a fact contains a specific numerical result, preserve that number.

Return ONLY valid JSON.

Required format:

{{
  "not_spurious": [
    "sentence 1",
    "sentence 2"
  ]
}}

FACTS:

{facts_text}
"""

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=4096
    ).to(llm.device)

    with torch.no_grad():
        outputs = llm.generate(
            **inputs,
            max_new_tokens=800,
            do_sample=True,
            temperature=0.5,
            top_p=0.8,
            top_k=20
        )

    generated_text = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    try:
        result = json.loads(generated_text)
    except json.JSONDecodeError:
        print("JSON ERROR")
        print(generated_text)
        return []

    rows = []

    for sentence in result.get("not_spurious", []):

        sentence = sentence.strip()

        if sentence:
            rows.append({
                "abs_id": abs_id,
                "sentence": sentence,
                "label": 0,
                "source": "synthetic_qwen"
            })

    return rows

In [ ]:
synthetic_test = paraphrase_facts(
    abstracts.iloc[0]["abs_id"],
    facts,
    num_sentences=10
)

synthetic_test_df = pd.DataFrame(synthetic_test)

print("Generated:", len(synthetic_test_df))

display(synthetic_test_df)

In [ ]:
def verify_sentence(abstract, sentence):

    prompt = f"""
You are a strict scientific fact-checker.

Determine whether the candidate sentence is explicitly supported by
the scientific abstract.

IMPORTANT:
- Use ONLY the information in the abstract.
- Do not use outside knowledge.
- The sentence must not introduce new facts.
- The sentence must not strengthen or weaken the original claim.
- Numerical values, models, methods, datasets and results must match.
- If the sentence makes a comparison, the abstract must explicitly
  support that comparison.
- If there is any uncertainty, mark it as NOT_SUPPORTED.

Return ONLY valid JSON.

Required format:

{{
  "supported": true
}}

or

{{
  "supported": false
}}

ABSTRACT:

{abstract}

CANDIDATE SENTENCE:

{sentence}
"""

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=4096
    ).to(llm.device)

    with torch.no_grad():
        outputs = llm.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False
        )

    generated_text = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    try:
        result = json.loads(generated_text)
        return bool(result.get("supported", False))

    except json.JSONDecodeError:
        return False

In [ ]:
# ============================================
# PRODUCTION SYNTHETIC DATA GENERATION
# ============================================

import os
import json
import pandas as pd

OUTPUT_DIR = "synthetic_data_production"
os.makedirs(OUTPUT_DIR, exist_ok=True)

FACTS_FILE = os.path.join(
    OUTPUT_DIR,
    "facts_cache.jsonl"
)

CANDIDATES_FILE = os.path.join(
    OUTPUT_DIR,
    "synthetic_candidates.jsonl"
)

VERIFIED_FILE = os.path.join(
    OUTPUT_DIR,
    "verified_synthetic.jsonl"
)

print("Production directory:", OUTPUT_DIR)
print("Facts cache:", FACTS_FILE)
print("Candidates:", CANDIDATES_FILE)
print("Verified:", VERIFIED_FILE)

Verification

In [ ]:
# ============================================
# BATCH VERIFIER
# ============================================

def verify_sentences_batch(abstract, sentences, max_retries=3):

    numbered_sentences = "\n".join(
        f"{i+1}. {sentence}"
        for i, sentence in enumerate(sentences)
    )

    prompt = f"""
You are a strict scientific fact-checker.

You are given one scientific abstract and several candidate sentences.

For EACH candidate sentence, determine whether it is explicitly
supported by the abstract.

Rules:

1. Use ONLY information explicitly stated in the abstract.
2. Do not use outside knowledge.
3. Do not infer information.
4. Do not introduce new methods, datasets, experiments, results,
   locations, numbers, or claims.
5. Do not strengthen or exaggerate the original claim.
6. A paraphrase is acceptable if it preserves the meaning.
7. If a sentence is uncertain or only partially supported, mark it false.
8. The sentence must be independently supported by the abstract.

IMPORTANT:
There must be exactly one boolean value for each candidate sentence.

Return ONLY valid JSON:

{{
    "supported": [true, false, true]
}}

The number of boolean values MUST be exactly {len(sentences)}.

ABSTRACT:
{abstract}

CANDIDATE SENTENCES:

{numbered_sentences}
"""

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    for attempt in range(1, max_retries + 1):

        try:

            text = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=False
            )

            inputs = tokenizer(
                text,
                return_tensors="pt",
                truncation=True,
                max_length=4096
            ).to(llm.device)

            with torch.no_grad():

                outputs = llm.generate(
                    **inputs,
                    max_new_tokens=100,
                    do_sample=False
                )

            generated_text = tokenizer.decode(
                outputs[0][inputs["input_ids"].shape[1]:],
                skip_special_tokens=True
            ).strip()

            # Find JSON inside the model response
            start = generated_text.find("{")
            end = generated_text.rfind("}")

            if start == -1 or end == -1:
                raise ValueError("No JSON object found")

            json_text = generated_text[start:end + 1]

            result = json.loads(json_text)

            supported = result.get("supported")

            if not isinstance(supported, list):
                raise ValueError(
                    "'supported' is not a list"
                )

            if len(supported) != len(sentences):
                raise ValueError(
                    f"Expected {len(sentences)} results, "
                    f"but received {len(supported)}"
                )

            if not all(
                isinstance(x, bool)
                for x in supported
            ):
                raise ValueError(
                    "Results must contain only true/false values"
                )

            # SUCCESS
            return supported

        except Exception as e:

            print(
                f"Verification attempt "
                f"{attempt}/{max_retries} failed: {e}"
            )

            if attempt < max_retries:
                print("Retrying...")

    # IMPORTANT:
    # A technical failure is NOT a rejection.
    return None

Generators

In [ ]:
# ============================================
# LOAD QWEN3-8B
# ============================================

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

QWEN_MODEL = "Qwen/Qwen3-8B"

print("Loading:", QWEN_MODEL)

qwen_tokenizer = AutoTokenizer.from_pretrained(
    QWEN_MODEL
)

qwen_model = AutoModelForCausalLM.from_pretrained(
    QWEN_MODEL,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("\nQwen loaded successfully!")
print("Device:", qwen_model.device)

print(
    "GPU memory allocated:",
    round(
        torch.cuda.memory_allocated() / 1024**3,
        2
    ),
    "GB"
)

In [ ]:
# ============================================
# QWEN3 — FINAL SYNTHETIC DATA GENERATION
# FACT EXTRACTION → PARAPHRASING
# ============================================

import os
import time
import pandas as pd

TARGET = 10_000
CHECKPOINT_EVERY = 500

QWEN_DIR = "./synthetic_qwen"
os.makedirs(QWEN_DIR, exist_ok=True)

CHECKPOINT_PATH = (
    f"{QWEN_DIR}/qwen_synthetic_10k_checkpoint.csv"
)

FINAL_PATH = (
    f"{QWEN_DIR}/qwen_synthetic_10k_raw.csv"
)

# --------------------------------------------
# Start / resume
# --------------------------------------------

if os.path.exists(CHECKPOINT_PATH):

    checkpoint_df = pd.read_csv(CHECKPOINT_PATH)

    qwen_results = checkpoint_df.to_dict("records")

    print(
        "Existing checkpoint:",
        len(qwen_results),
        "rows"
    )

else:

    qwen_results = []

    print("Starting new Qwen generation...")

# --------------------------------------------
# Generation
# --------------------------------------------

start_time = time.time()

abstract_index = 0

while len(qwen_results) < TARGET:

    row = abstracts.iloc[
        abstract_index % len(abstracts)
    ]

    abstract_index += 1

    abs_id = row["abs_id"]
    abstract = row["abs_source"]

    try:

        # ------------------------------------
        # Stage 1: Extract explicit facts
        # ------------------------------------

        facts = extract_explicit_facts(
            abs_id,
            abstract
        )

        if not facts:

            print(
                f"No facts extracted for {abs_id}"
            )
            continue

        # ------------------------------------
        # Stage 2: Generate paraphrases
        # ------------------------------------

        remaining = TARGET - len(qwen_results)

        num_sentences = min(
            10,
            remaining
        )

        generated_rows = paraphrase_facts(
            abs_id,
            facts,
            num_sentences=num_sentences
        )

        if not generated_rows:

            print(
                f"No synthetic sentences generated for {abs_id}"
            )
            continue

        # ------------------------------------
        # Add generated rows
        # ------------------------------------

        qwen_results.extend(
            generated_rows
        )

        # Safety: do not exceed target
        if len(qwen_results) > TARGET:

            qwen_results = qwen_results[:TARGET]

    except Exception as e:

        print(
            f"Generation error for {abs_id}: {e}"
        )
        continue

    completed = len(qwen_results)

    # ----------------------------------------
    # Checkpoint
    # ----------------------------------------

    if (
        completed % CHECKPOINT_EVERY == 0
        or completed == TARGET
    ):

        checkpoint_df = pd.DataFrame(
            qwen_results
        )

        checkpoint_df.to_csv(
            CHECKPOINT_PATH,
            index=False
        )

        elapsed = time.time() - start_time

        print(
            f"Generated {completed}/{TARGET} "
            f"| elapsed: {elapsed/60:.1f} min "
            f"| checkpoint saved"
        )

# --------------------------------------------
# Final save
# --------------------------------------------

qwen_synthetic_10k_df = pd.DataFrame(
    qwen_results
)

qwen_synthetic_10k_df.to_csv(
    FINAL_PATH,
    index=False
)

elapsed = time.time() - start_time

print("\n============================================")
print("QWEN SYNTHETIC GENERATION COMPLETE")
print("============================================")

print(
    "Rows:",
    len(qwen_synthetic_10k_df)
)

print(
    "Unique abstracts:",
    qwen_synthetic_10k_df["abs_id"].nunique()
)

print(
    "Duplicate sentences:",
    qwen_synthetic_10k_df["sentence"]
    .duplicated()
    .sum()
)

print(
    "Time:",
    round(elapsed / 60, 2),
    "minutes"
)

print(
    "Saved:",
    FINAL_PATH
)

display(
    qwen_synthetic_10k_df.head()
)

In [ ]:
# ============================================
# QWEN 10K — REMOVE DUPLICATES AND OVERLAP
# ============================================

import pandas as pd

QWEN_RAW_PATH = (
    "./synthetic_qwen/"
    "qwen_synthetic_10k_raw.csv"
)

QWEN_UNIQUE_PATH = (
    "./synthetic_qwen/"
    "qwen_synthetic_10k_unique.csv"
)

# --------------------------------------------
# Load generated data
# --------------------------------------------

qwen_10k_raw = pd.read_csv(
    QWEN_RAW_PATH
)

print("Raw rows:", len(qwen_10k_raw))

# --------------------------------------------
# Remove duplicate synthetic sentences
# --------------------------------------------

qwen_10k_unique = (
    qwen_10k_raw
    .drop_duplicates(
        subset=["sentence"],
        keep="first"
    )
    .reset_index(drop=True)
)

print(
    "After synthetic deduplication:",
    len(qwen_10k_unique)
)

print(
    "Synthetic duplicates removed:",
    len(qwen_10k_raw)
    - len(qwen_10k_unique)
)

# --------------------------------------------
# Remove overlap with original training data
# --------------------------------------------

training_sentences = set(
    train_df["sentence"]
    .dropna()
    .astype(str)
    .str.strip()
)

before_overlap_removal = len(
    qwen_10k_unique
)

qwen_10k_unique = qwen_10k_unique[
    ~qwen_10k_unique["sentence"]
    .astype(str)
    .str.strip()
    .isin(training_sentences)
].reset_index(drop=True)

overlap_removed = (
    before_overlap_removal
    - len(qwen_10k_unique)
)

print(
    "Original-training overlap removed:",
    overlap_removed
)

# --------------------------------------------
# Final checks
# --------------------------------------------

print(
    "Final unique rows:",
    len(qwen_10k_unique)
)

print(
    "Unique abstracts:",
    qwen_10k_unique["abs_id"].nunique()
)

print(
    "Remaining duplicate sentences:",
    qwen_10k_unique["sentence"]
    .duplicated()
    .sum()
)

# --------------------------------------------
# Save
# --------------------------------------------

qwen_10k_unique.to_csv(
    QWEN_UNIQUE_PATH,
    index=False
)

print(
    "Saved:",
    QWEN_UNIQUE_PATH
)

display(
    qwen_10k_unique.head()
)

In [ ]:
# ============================================
# QWEN — PREPARE FOR VERIFICATION
# ============================================

QWEN_UNIQUE_PATH = (
    "./synthetic_qwen/"
    "qwen_synthetic_10k_unique.csv"
)

qwen_verify_df = pd.read_csv(
    QWEN_UNIQUE_PATH
)

print(
    "Qwen candidates to verify:",
    len(qwen_verify_df)
)

print(
    "Verifier available:",
    "verify_sentences_batch" in globals()
)

print(
    "Source abstracts available:",
    "abstracts" in globals()
)

In [ ]:
# ============================================
# RESTORE ROBUST QWEN VERIFIER
# ============================================

import json
import re
import torch


# Use the already-loaded Qwen model as verifier
verifier_model = qwen_model
verifier_tokenizer = qwen_tokenizer


def verify_sentences_batch(abstract, sentences):

    numbered_sentences = "\n".join(
        f"{i+1}. {sentence}"
        for i, sentence in enumerate(sentences)
    )

    prompt = f"""
You are a strict scientific fact-checker.

You are given one scientific abstract and several candidate sentences.
For EACH candidate sentence, determine whether it is explicitly
supported by the abstract.

Rules:
1. Use ONLY information explicitly stated in the abstract.
2. Do not use outside knowledge.
3. Do not infer information.
4. Do not introduce new methods, datasets, experiments, results,
   locations, numbers, or claims.
5. Do not strengthen or exaggerate the original claim.
6. A paraphrase is acceptable if it preserves the meaning.
7. If a sentence is uncertain or only partially supported, mark it false.
8. The sentence must be independently supported by the abstract.

Return ONLY valid JSON in exactly this format:

{{
  "supported": [true, false, true]
}}

There must be exactly one boolean value for each candidate sentence,
in the same order.

ABSTRACT:
{abstract}

CANDIDATE SENTENCES:
{numbered_sentences}
"""

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    try:
        text = verifier_tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False
        )
    except TypeError:
        text = verifier_tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

    inputs = verifier_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=4096
    ).to(verifier_model.device)

    with torch.no_grad():

        outputs = verifier_model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False
        )

    generated_text = verifier_tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    # ----------------------------------------
    # Robust JSON extraction
    # ----------------------------------------

    try:

        # First try the complete output
        try:
            result = json.loads(generated_text)

        except json.JSONDecodeError:

            # Extract the JSON object even if Qwen
            # adds explanation/reasoning afterwards
            match = re.search(
                r'\{\s*"supported"\s*:\s*\[[\s\S]*?\]\s*\}',
                generated_text
            )

            if not match:
                raise ValueError(
                    "No valid supported JSON found"
                )

            result = json.loads(
                match.group(0)
            )

        supported = result["supported"]

        if len(supported) != len(sentences):

            print(
                "WARNING: Incorrect number of "
                "verification results."
            )

            return [False] * len(sentences)

        return [
            bool(x)
            for x in supported
        ]

    except Exception as e:

        print(
            "Verification parsing error:",
            e
        )

        print(
            "Model output:"
        )

        print(generated_text)

        # Fail-safe
        return [False] * len(sentences)


print("Robust Qwen verifier restored successfully!")
print(
    "Verifier model:",
    type(verifier_model).__name__
)
print(
    "Verifier tokenizer:",
    type(verifier_tokenizer).__name__
)

In [ ]:
# ============================================
# QWEN — FULL 3,216 CANDIDATE VERIFICATION
# ============================================

import os
import time
import pandas as pd

INPUT_PATH = (
    "./synthetic_qwen/"
    "qwen_synthetic_10k_unique.csv"
)

CHECKPOINT_PATH = (
    "./synthetic_qwen/"
    "qwen_synthetic_10k_verified_checkpoint.csv"
)

FINAL_PATH = (
    "./synthetic_qwen/"
    "qwen_synthetic_10k_verified.csv"
)

BATCH_SIZE = 10
CHECKPOINT_EVERY = 100

df = pd.read_csv(INPUT_PATH)

print("Total unique candidates:", len(df))

# --------------------------------------------
# Resume if checkpoint exists
# --------------------------------------------

if os.path.exists(CHECKPOINT_PATH):

    verified_df = pd.read_csv(
        CHECKPOINT_PATH
    )

    start_idx = len(verified_df)

    print(
        "Existing verification checkpoint:",
        start_idx
    )

else:

    verified_df = df.iloc[0:0].copy()
    verified_df["verified"] = pd.Series(
        dtype="bool"
    )

    start_idx = 0

    print("Starting verification from 0")

# --------------------------------------------
# Verification loop
# --------------------------------------------

start_time = time.time()

for start in range(
    start_idx,
    len(df),
    BATCH_SIZE
):

    batch = df.iloc[
        start:start + BATCH_SIZE
    ].copy()

    batch_results = []

    for abs_id, group in batch.groupby(
        "abs_id",
        sort=False
    ):

        abstract_matches = abstracts.loc[
            abstracts["abs_id"] == abs_id,
            "abs_source"
        ]

        if len(abstract_matches) == 0:

            print(
                f"WARNING: Abstract not found: {abs_id}"
            )

            batch_results.extend(
                [False] * len(group)
            )

            continue

        abstract = abstract_matches.iloc[0]

        sentences = group["sentence"].tolist()

        try:

            results = verify_sentences_batch(
                abstract,
                sentences
            )

            if len(results) != len(sentences):

                print(
                    "WARNING: Incorrect number "
                    "of verification results for",
                    abs_id
                )

                results = [False] * len(sentences)

        except Exception as e:

            print(
                f"Verification error for {abs_id}: {e}"
            )

            results = [False] * len(sentences)

        batch_results.extend(results)

    batch["verified"] = batch_results

    verified_df = pd.concat(
        [verified_df, batch],
        ignore_index=True
    )

    completed = len(verified_df)

    if (
        completed % CHECKPOINT_EVERY == 0
        or completed >= len(df)
    ):

        verified_df.to_csv(
            CHECKPOINT_PATH,
            index=False
        )

        elapsed = time.time() - start_time

        print(
            f"Verified {completed}/{len(df)} "
            f"| elapsed: {elapsed/60:.1f} min "
            f"| checkpoint saved"
        )

# ============================================
# Final save
# ============================================

verified_df.to_csv(
    FINAL_PATH,
    index=False
)

elapsed = time.time() - start_time

print("\n============================================")
print("QWEN VERIFICATION COMPLETE")
print("============================================")

print(
    "Total candidates:",
    len(verified_df)
)

print("\nVerification distribution:")
print(
    verified_df["verified"].value_counts()
)

print("\nVerification percentages:")
print(
    verified_df["verified"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print(
    "\nVerified:",
    int(verified_df["verified"].sum())
)

print(
    "Rejected:",
    int((~verified_df["verified"]).sum())
)

print(
    "Time:",
    round(elapsed / 60, 2),
    "minutes"
)

print(
    "Saved:",
    FINAL_PATH
)

display(
    verified_df.head()
)

In [ ]:
# ============================================
# FINAL SYNTHETIC DATA SANITY CHECK
# ============================================

import pandas as pd
import os

# --------------------------------------------
# Load verified datasets
# --------------------------------------------

QWEN_PATH = "./synthetic_qwen/qwen_synthetic_10k_verified.csv"
LLAMA_PATH = "./synthetic_llama/llama_synthetic_10k_verified.csv"
MISTRAL_PATH = "./synthetic_mistral/mistral_synthetic_10k_verified.csv"

qwen_verified = pd.read_csv(QWEN_PATH)
llama_verified = pd.read_csv(LLAMA_PATH)
mistral_verified = pd.read_csv(MISTRAL_PATH)

# Keep only accepted examples
qwen_accepted = qwen_verified[
    qwen_verified["verified"] == True
].copy()

llama_accepted = llama_verified[
    llama_verified["verified"] == True
].copy()

mistral_accepted = mistral_verified[
    mistral_verified["verified"] == True
].copy()

print("============================================")
print("VERIFIED SYNTHETIC DATA")
print("============================================")

print("\nQwen:")
print("  Verified:", len(qwen_accepted))
print("  Unique sentences:",
      qwen_accepted["sentence"].nunique())
print("  Unique abstracts:",
      qwen_accepted["abs_id"].nunique())

print("\nLlama:")
print("  Verified:", len(llama_accepted))
print("  Unique sentences:",
      llama_accepted["sentence"].nunique())
print("  Unique abstracts:",
      llama_accepted["abs_id"].nunique())

print("\nMistral:")
print("  Verified:", len(mistral_accepted))
print("  Unique sentences:",
      mistral_accepted["sentence"].nunique())
print("  Unique abstracts:",
      mistral_accepted["abs_id"].nunique())


# ============================================
# CHECK LABELS
# ============================================

print("\n============================================")
print("LABEL CHECK")
print("============================================")

for name, data in [
    ("Qwen", qwen_accepted),
    ("Llama", llama_accepted),
    ("Mistral", mistral_accepted)
]:

    print(f"\n{name} labels:")
    print(data["label"].value_counts().to_dict())


# ============================================
# CHECK MISSING VALUES
# ============================================

print("\n============================================")
print("MISSING VALUE CHECK")
print("============================================")

for name, data in [
    ("Qwen", qwen_accepted),
    ("Llama", llama_accepted),
    ("Mistral", mistral_accepted)
]:

    print(f"\n{name}:")
    print(data[
        ["abs_id", "sentence", "label", "source"]
    ].isna().sum())


# ============================================
# CHECK DUPLICATES
# ============================================

print("\n============================================")
print("DUPLICATE CHECK")
print("============================================")

for name, data in [
    ("Qwen", qwen_accepted),
    ("Llama", llama_accepted),
    ("Mistral", mistral_accepted)
]:

    print(
        f"{name} duplicate sentences:",
        data["sentence"].duplicated().sum()
    )


# ============================================
# CHECK SOURCE ABSTRACTS
# ============================================

print("\n============================================")
print("SOURCE ABSTRACT CHECK")
print("============================================")

source_ids = set(
    abstracts["abs_id"]
)

for name, data in [
    ("Qwen", qwen_accepted),
    ("Llama", llama_accepted),
    ("Mistral", mistral_accepted)
]:

    missing_sources = set(
        data["abs_id"]
    ) - source_ids

    print(
        f"{name} missing source abstracts:",
        len(missing_sources)
    )


# ============================================
# FINAL SUMMARY
# ============================================

print("\n============================================")
print("FINAL SANITY CHECK COMPLETE")
print("============================================")

print(
    "\nCommon synthetic sample size available:",
    len(qwen_accepted)
)

print(
    "\nExpected controlled comparison:"
)

print("  Qwen    → 3,207 synthetic")
print("  Llama   → 3,207 synthetic")
print("  Mistral → 3,207 synthetic")

Llama

In [ ]:
# ============================================
# LOAD LLAMA 3.1 8B
# ============================================

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

LLAMA_MODEL = "meta-llama/Llama-3.1-8B-Instruct"

print("Loading:", LLAMA_MODEL)

llama_tokenizer = AutoTokenizer.from_pretrained(
    LLAMA_MODEL
)

llama_model = AutoModelForCausalLM.from_pretrained(
    LLAMA_MODEL,
    dtype=torch.float16,
    device_map="auto"
)

print("\nLlama loaded successfully!")
print("Device:", llama_model.device)

print(
    "GPU memory allocated:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

print(
    "GPU memory reserved:",
    round(torch.cuda.memory_reserved() / 1024**3, 2),
    "GB"
)

In [ ]:
# ============================================
# LLAMA 3.1-8B-INSTRUCT — FINAL SYNTHETIC DATA GENERATION
# FACT EXTRACTION → PARAPHRASING
# ============================================

import os
import time
import pandas as pd

TARGET = 10_000
CHECKPOINT_EVERY = 500

LLAMA_DIR = "./synthetic_llama"
os.makedirs(LLAMA_DIR, exist_ok=True)

CHECKPOINT_PATH = (
    f"{LLAMA_DIR}/llama_synthetic_10k_checkpoint.csv"
)

FINAL_PATH = (
    f"{LLAMA_DIR}/llama_synthetic_10k_raw.csv"
)

# --------------------------------------------
# Use Llama for the generation functions
# --------------------------------------------

llm = llama_model
tokenizer = llama_tokenizer

# --------------------------------------------
# Start / resume
# --------------------------------------------

if os.path.exists(CHECKPOINT_PATH):

    checkpoint_df = pd.read_csv(
        CHECKPOINT_PATH
    )

    llama_results = (
        checkpoint_df
        .to_dict("records")
    )

    print(
        "Existing checkpoint:",
        len(llama_results),
        "rows"
    )

else:

    llama_results = []

    print(
        "Starting new Llama generation..."
    )

# --------------------------------------------
# Generation
# --------------------------------------------

start_time = time.time()

abstract_index = 0

while len(llama_results) < TARGET:

    row = abstracts.iloc[
        abstract_index % len(abstracts)
    ]

    abstract_index += 1

    abs_id = row["abs_id"]
    abstract = row["abs_source"]

    try:

        # ------------------------------------
        # Stage 1: Extract explicit facts
        # ------------------------------------

        facts = extract_explicit_facts(
            abs_id,
            abstract
        )

        if not facts:

            print(
                f"No facts extracted for {abs_id}"
            )
            continue

        # ------------------------------------
        # Stage 2: Generate paraphrases
        # ------------------------------------

        remaining = (
            TARGET - len(llama_results)
        )

        num_sentences = min(
            10,
            remaining
        )

        generated_rows = paraphrase_facts(
            abs_id,
            facts,
            num_sentences=num_sentences
        )

        if not generated_rows:

            print(
                f"No synthetic sentences generated for {abs_id}"
            )
            continue

        # ------------------------------------
        # Add generated rows
        # ------------------------------------

        llama_results.extend(
            generated_rows
        )

        # Safety: do not exceed target
        if len(llama_results) > TARGET:

            llama_results = (
                llama_results[:TARGET]
            )

    except Exception as e:

        print(
            f"Generation error for {abs_id}: {e}"
        )
        continue

    completed = len(llama_results)

    # ----------------------------------------
    # Checkpoint
    # ----------------------------------------

    if (
        completed % CHECKPOINT_EVERY == 0
        or completed == TARGET
    ):

        checkpoint_df = pd.DataFrame(
            llama_results
        )

        checkpoint_df.to_csv(
            CHECKPOINT_PATH,
            index=False
        )

        elapsed = (
            time.time() - start_time
        )

        print(
            f"Generated {completed}/{TARGET} "
            f"| elapsed: {elapsed/60:.1f} min "
            f"| checkpoint saved"
        )

# --------------------------------------------
# Final save
# --------------------------------------------

llama_synthetic_5k_df = pd.DataFrame(
    llama_results
)

llama_synthetic_5k_df.to_csv(
    FINAL_PATH,
    index=False
)

elapsed = (
    time.time() - start_time
)

print("\n============================================")
print("LLAMA SYNTHETIC GENERATION COMPLETE")
print("============================================")

print(
    "Rows:",
    len(llama_synthetic_5k_df)
)

print(
    "Unique abstracts:",
    llama_synthetic_5k_df["abs_id"].nunique()
)

print(
    "Duplicate sentences:",
    llama_synthetic_5k_df["sentence"]
    .duplicated()
    .sum()
)

print(
    "Time:",
    round(elapsed / 60, 2),
    "minutes"
)

print(
    "Saved:",
    FINAL_PATH
)

display(
    llama_synthetic_5k_df.head()
)

In [ ]:
# ============================================
# LLAMA 10K — REMOVE DUPLICATES AND OVERLAP
# ============================================

import pandas as pd

LLAMA_RAW_PATH = (
    "./synthetic_llama/"
    "llama_synthetic_10k_raw.csv"
)

LLAMA_UNIQUE_PATH = (
    "./synthetic_llama/"
    "llama_synthetic_10k_unique.csv"
)

# --------------------------------------------
# Load generated data
# --------------------------------------------

llama_10k_raw = pd.read_csv(
    LLAMA_RAW_PATH
)

print("Raw rows:", len(llama_10k_raw))

# --------------------------------------------
# Remove duplicate synthetic sentences
# --------------------------------------------

llama_10k_unique = (
    llama_10k_raw
    .drop_duplicates(
        subset=["sentence"],
        keep="first"
    )
    .reset_index(drop=True)
)

print(
    "After synthetic deduplication:",
    len(llama_10k_unique)
)

print(
    "Synthetic duplicates removed:",
    len(llama_10k_raw)
    - len(llama_10k_unique)
)

# --------------------------------------------
# Remove overlap with original training data
# --------------------------------------------

training_sentences = set(
    train_df["sentence"]
    .dropna()
    .astype(str)
    .str.strip()
)

before_overlap_removal = len(
    llama_10k_unique
)

llama_10k_unique = llama_10k_unique[
    ~llama_10k_unique["sentence"]
    .astype(str)
    .str.strip()
    .isin(training_sentences)
].reset_index(drop=True)

overlap_removed = (
    before_overlap_removal
    - len(llama_10k_unique)
)

print(
    "Original-training overlap removed:",
    overlap_removed
)

# --------------------------------------------
# Final checks
# --------------------------------------------

print(
    "Final unique rows:",
    len(llama_10k_unique)
)

print(
    "Unique abstracts:",
    llama_10k_unique["abs_id"].nunique()
)

print(
    "Remaining duplicate sentences:",
    llama_10k_unique["sentence"]
    .duplicated()
    .sum()
)

# --------------------------------------------
# Save
# --------------------------------------------

llama_10k_unique.to_csv(
    LLAMA_UNIQUE_PATH,
    index=False
)

print(
    "Saved:",
    LLAMA_UNIQUE_PATH
)

display(
    llama_10k_unique.head()
)

In [ ]:
# ============================================
# LLAMA 10K UNIQUE — SOURCE VERIFICATION
# ============================================

import pandas as pd
import time
import json

LLAMA_UNIQUE_PATH = (
    "./synthetic_llama/"
    "llama_synthetic_10k_unique.csv"
)

llama_verify_df = pd.read_csv(
    LLAMA_UNIQUE_PATH
)

print(
    "Candidates to verify:",
    len(llama_verify_df)
)

# Make sure verifier exists
print(
    "Verifier available:",
    "verify_sentence" in globals()
)

In [ ]:
# ============================================
# LLAMA CANDIDATE VERIFICATION
# ============================================

import os
import time
import pandas as pd

INPUT_PATH = (
    "./synthetic_llama/"
    "llama_synthetic_10k_unique.csv"
)

CHECKPOINT_PATH = (
    "./synthetic_llama/"
    "llama_synthetic_10k_verified_checkpoint.csv"
)

FINAL_PATH = (
    "./synthetic_llama/"
    "llama_synthetic_10k_verified.csv"
)

BATCH_SIZE = 10
CHECKPOINT_EVERY = 100

df = pd.read_csv(INPUT_PATH)

print("Total unique candidates:", len(df))

# --------------------------------------------
# Resume if checkpoint exists
# --------------------------------------------

if os.path.exists(CHECKPOINT_PATH):

    verified_df = pd.read_csv(
        CHECKPOINT_PATH
    )

    start_idx = len(verified_df)

    print(
        "Existing verification checkpoint:",
        start_idx
    )

else:

    verified_df = df.iloc[0:0].copy()
    verified_df["verified"] = pd.Series(
        dtype="bool"
    )

    start_idx = 0

    print("Starting verification from 0")

# --------------------------------------------
# Verification loop
# --------------------------------------------

start_time = time.time()

for start in range(
    start_idx,
    len(df),
    BATCH_SIZE
):

    batch = df.iloc[
        start:start + BATCH_SIZE
    ].copy()

    batch_results = []

    # Group by abstract so each abstract is
    # verified with all its candidate sentences
    for abs_id, group in batch.groupby(
        "abs_id",
        sort=False
    ):

        abstract_matches = abstracts.loc[
            abstracts["abs_id"] == abs_id,
            "abs_source"
        ]

        if len(abstract_matches) == 0:

            print(
                f"WARNING: Abstract not found: {abs_id}"
            )

            batch_results.extend(
                [False] * len(group)
            )

            continue

        abstract = abstract_matches.iloc[0]

        sentences = group["sentence"].tolist()

        try:

            results = verify_sentences_batch(
                abstract,
                sentences
            )

            if len(results) != len(sentences):

                print(
                    "WARNING: Incorrect number "
                    "of verification results for",
                    abs_id
                )

                results = [False] * len(sentences)

        except Exception as e:

            print(
                f"Verification error for {abs_id}: {e}"
            )

            results = [False] * len(sentences)

        batch_results.extend(results)

    # ----------------------------------------
    # Add verification results
    # ----------------------------------------

    batch["verified"] = batch_results

    verified_df = pd.concat(
        [verified_df, batch],
        ignore_index=True
    )

    completed = len(verified_df)

    # ----------------------------------------
    # Checkpoint
    # ----------------------------------------

    if (
        completed % CHECKPOINT_EVERY == 0
        or completed >= len(df)
    ):

        verified_df.to_csv(
            CHECKPOINT_PATH,
            index=False
        )

        elapsed = time.time() - start_time

        print(
            f"Verified {completed}/{len(df)} "
            f"| elapsed: {elapsed/60:.1f} min "
            f"| checkpoint saved"
        )

# ============================================
# Final save
# ============================================

verified_df.to_csv(
    FINAL_PATH,
    index=False
)

elapsed = time.time() - start_time

print("\n============================================")
print("LLAMA VERIFICATION COMPLETE")
print("============================================")

print(
    "Total candidates:",
    len(verified_df)
)

print("\nVerification distribution:")
print(
    verified_df["verified"].value_counts()
)

print("\nVerification percentages:")
print(
    verified_df["verified"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print(
    "\nVerified:",
    int(verified_df["verified"].sum())
)

print(
    "Rejected:",
    int((~verified_df["verified"]).sum())
)

print(
    "Time:",
    round(elapsed / 60, 2),
    "minutes"
)

print(
    "Saved:",
    FINAL_PATH
)

display(
    verified_df.head()
)

Mistral

In [ ]:
# ============================================
# LOAD MISTRAL 7B
# ============================================

from transformers import AutoTokenizer, AutoModelForCausalLM

MISTRAL_MODEL = "mistralai/Mistral-7B-Instruct-v0.3"

print("Loading:", MISTRAL_MODEL)

mistral_tokenizer = AutoTokenizer.from_pretrained(
    MISTRAL_MODEL
)

mistral_model = AutoModelForCausalLM.from_pretrained(
    MISTRAL_MODEL,
    dtype=torch.float16,
    device_map="auto"
)

print("\nMistral loaded successfully!")
print("Device:", mistral_model.device)

print(
    "GPU memory allocated:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

In [ ]:
# ============================================
# MISTRAL-7B-INSTRUCT-V0.3 — FINAL SYNTHETIC DATA GENERATION
# FACT EXTRACTION → PARAPHRASING
# ============================================

import os
import time
import pandas as pd

TARGET = 10_000
CHECKPOINT_EVERY = 500

MISTRAL_DIR = "./synthetic_mistral"
os.makedirs(MISTRAL_DIR, exist_ok=True)

CHECKPOINT_PATH = (
    f"{MISTRAL_DIR}/mistral_synthetic_10k_checkpoint.csv"
)

FINAL_PATH = (
    f"{MISTRAL_DIR}/mistral_synthetic_10k_raw.csv"
)

# --------------------------------------------
# Use Mistral for the generation functions
# --------------------------------------------

llm = mistral_model
tokenizer = mistral_tokenizer

# --------------------------------------------
# Start / resume
# --------------------------------------------

if os.path.exists(CHECKPOINT_PATH):

    checkpoint_df = pd.read_csv(
        CHECKPOINT_PATH
    )

    mistral_results = (
        checkpoint_df
        .to_dict("records")
    )

    print(
        "Existing checkpoint:",
        len(mistral_results),
        "rows"
    )

else:

    mistral_results = []

    print(
        "Starting new Mistral generation..."
    )

# --------------------------------------------
# Generation
# --------------------------------------------

start_time = time.time()

abstract_index = 0

while len(mistral_results) < TARGET:

    row = abstracts.iloc[
        abstract_index % len(abstracts)
    ]

    abstract_index += 1

    abs_id = row["abs_id"]
    abstract = row["abs_source"]

    try:

        # ------------------------------------
        # Stage 1: Extract explicit facts
        # ------------------------------------

        facts = extract_explicit_facts(
            abs_id,
            abstract
        )

        if not facts:

            print(
                f"No facts extracted for {abs_id}"
            )
            continue

        # ------------------------------------
        # Stage 2: Generate paraphrases
        # ------------------------------------

        remaining = (
            TARGET - len(mistral_results)
        )

        num_sentences = min(
            10,
            remaining
        )

        generated_rows = paraphrase_facts(
            abs_id,
            facts,
            num_sentences=num_sentences
        )

        if not generated_rows:

            print(
                f"No synthetic sentences generated for {abs_id}"
            )
            continue

        # ------------------------------------
        # Add generated rows
        # ------------------------------------

        mistral_results.extend(
            generated_rows
        )

        # Safety: do not exceed target
        if len(mistral_results) > TARGET:

            mistral_results = (
                mistral_results[:TARGET]
            )

    except Exception as e:

        print(
            f"Generation error for {abs_id}: {e}"
        )
        continue

    completed = len(mistral_results)

    # ----------------------------------------
    # Checkpoint
    # ----------------------------------------

    if (
        completed % CHECKPOINT_EVERY == 0
        or completed == TARGET
    ):

        checkpoint_df = pd.DataFrame(
            mistral_results
        )

        checkpoint_df.to_csv(
            CHECKPOINT_PATH,
            index=False
        )

        elapsed = (
            time.time() - start_time
        )

        print(
            f"Generated {completed}/{TARGET} "
            f"| elapsed: {elapsed/60:.1f} min "
            f"| checkpoint saved"
        )

# --------------------------------------------
# Final save
# --------------------------------------------

mistral_synthetic_10k_df = pd.DataFrame(
    mistral_results
)

mistral_synthetic_10k_df.to_csv(
    FINAL_PATH,
    index=False
)

elapsed = (
    time.time() - start_time
)

print("\n============================================")
print("MISTRAL SYNTHETIC GENERATION COMPLETE")
print("============================================")

print(
    "Rows:",
    len(mistral_synthetic_10k_df)
)

print(
    "Unique abstracts:",
    mistral_synthetic_10k_df["abs_id"].nunique()
)

print(
    "Duplicate sentences:",
    mistral_synthetic_10k_df["sentence"]
    .duplicated()
    .sum()
)

print(
    "Time:",
    round(elapsed / 60, 2),
    "minutes"
)

print(
    "Saved:",
    FINAL_PATH
)

display(
    mistral_synthetic_10k_df.head()
)

In [ ]:
# ============================================
# MISTRAL 10K — REMOVE DUPLICATES AND OVERLAP
# ============================================

import pandas as pd

MISTRAL_RAW_PATH = (
    "./synthetic_mistral/"
    "mistral_synthetic_10k_raw.csv"
)

MISTRAL_UNIQUE_PATH = (
    "./synthetic_mistral/"
    "mistral_synthetic_10k_unique.csv"
)

# --------------------------------------------
# Load generated data
# --------------------------------------------

mistral_10k_raw = pd.read_csv(
    MISTRAL_RAW_PATH
)

print("Raw rows:", len(mistral_10k_raw))

# --------------------------------------------
# Remove duplicate synthetic sentences
# --------------------------------------------

mistral_10k_unique = (
    mistral_10k_raw
    .drop_duplicates(
        subset=["sentence"],
        keep="first"
    )
    .reset_index(drop=True)
)

print(
    "After synthetic deduplication:",
    len(mistral_10k_unique)
)

print(
    "Synthetic duplicates removed:",
    len(mistral_10k_raw)
    - len(mistral_10k_unique)
)

# --------------------------------------------
# Remove overlap with original training data
# --------------------------------------------

training_sentences = set(
    train_df["sentence"]
    .dropna()
    .astype(str)
    .str.strip()
)

before_overlap_removal = len(
    mistral_10k_unique
)

mistral_10k_unique = mistral_10k_unique[
    ~mistral_10k_unique["sentence"]
    .astype(str)
    .str.strip()
    .isin(training_sentences)
].reset_index(drop=True)

overlap_removed = (
    before_overlap_removal
    - len(mistral_10k_unique)
)

print(
    "Original-training overlap removed:",
    overlap_removed
)

# --------------------------------------------
# Final checks
# --------------------------------------------

print(
    "Final unique rows:",
    len(mistral_10k_unique)
)

print(
    "Unique abstracts:",
    mistral_10k_unique["abs_id"].nunique()
)

print(
    "Remaining duplicate sentences:",
    mistral_10k_unique["sentence"]
    .duplicated()
    .sum()
)

# --------------------------------------------
# Save
# --------------------------------------------

mistral_10k_unique.to_csv(
    MISTRAL_UNIQUE_PATH,
    index=False
)

print(
    "Saved:",
    MISTRAL_UNIQUE_PATH
)

display(
    mistral_10k_unique.head()
)

In [ ]:
# ============================================
# LOAD MISTRAL UNIQUE CANDIDATES
# ============================================

MISTRAL_UNIQUE_PATH = (
    "./synthetic_mistral/"
    "mistral_synthetic_10k_unique.csv"
)

mistral_verify_df = pd.read_csv(
    MISTRAL_UNIQUE_PATH
)

print(
    "Mistral candidates to verify:",
    len(mistral_verify_df)
)

print(
    "Verifier available:",
    "verify_sentences_batch" in globals()
)

In [ ]:
# ============================================
# MISTRAL — FULL UNIQUE CANDIDATE VERIFICATION
# ============================================

import os
import time
import pandas as pd

INPUT_PATH = (
    "./synthetic_mistral/"
    "mistral_synthetic_10k_unique.csv"
)

CHECKPOINT_PATH = (
    "./synthetic_mistral/"
    "mistral_synthetic_10k_verified_checkpoint.csv"
)

FINAL_PATH = (
    "./synthetic_mistral/"
    "mistral_synthetic_10k_verified.csv"
)

BATCH_SIZE = 10
CHECKPOINT_EVERY = 100

df = pd.read_csv(INPUT_PATH)

print("Total unique candidates:", len(df))

# ============================================
# LOAD / RESUME CHECKPOINT
# ============================================

if os.path.exists(CHECKPOINT_PATH):

    verified_df = pd.read_csv(
        CHECKPOINT_PATH
    )

    start_idx = len(verified_df)

    print(
        "Existing verification checkpoint:",
        start_idx
    )

    if start_idx > len(df):
        raise ValueError(
            "Checkpoint contains more rows than input."
        )

    # Make sure checkpoint is aligned
    if start_idx > 0:

        checkpoint_ids = (
            verified_df["abs_id"].tolist()
        )

        input_ids = (
            df.iloc[:start_idx]["abs_id"].tolist()
        )

        if checkpoint_ids != input_ids:
            raise ValueError(
                "Checkpoint does not match input ordering."
            )

        print("Checkpoint alignment: OK")

else:

    verified_df = df.iloc[0:0].copy()

    verified_df["verified"] = pd.Series(
        dtype="bool"
    )

    start_idx = 0

    print("Starting verification from 0")

# ============================================
# VERIFICATION LOOP
# ============================================

start_time = time.time()

for start in range(
    start_idx,
    len(df),
    BATCH_SIZE
):

    batch = df.iloc[
        start:start + BATCH_SIZE
    ].copy()

    batch_results = []

    for abs_id, group in batch.groupby(
        "abs_id",
        sort=False
    ):

        abstract_matches = abstracts.loc[
            abstracts["abs_id"] == abs_id,
            "abs_source"
        ]

        if len(abstract_matches) == 0:

            print(
                f"WARNING: Abstract not found: {abs_id}"
            )

            batch_results.extend(
                [False] * len(group)
            )

            continue

        abstract = abstract_matches.iloc[0]

        sentences = group["sentence"].tolist()

        try:

            results = verify_sentences_batch(
                abstract,
                sentences
            )

            if len(results) != len(sentences):

                print(
                    f"WARNING: Expected "
                    f"{len(sentences)} results but got "
                    f"{len(results)} for {abs_id}"
                )

                results = [False] * len(sentences)

        except Exception as e:

            print(
                f"Verification error for {abs_id}: {e}"
            )

            results = [False] * len(sentences)

        batch_results.extend(results)

    # ----------------------------------------
    # Safety check
    # ----------------------------------------

    if len(batch_results) != len(batch):

        raise RuntimeError(
            f"Batch mismatch: "
            f"{len(batch_results)} results for "
            f"{len(batch)} candidates."
        )

    batch["verified"] = batch_results

    verified_df = pd.concat(
        [verified_df, batch],
        ignore_index=True
    )

    completed = len(verified_df)

    # ========================================
    # CHECKPOINT
    # ========================================

    if (
        completed % CHECKPOINT_EVERY == 0
        or completed == len(df)
    ):

        verified_df.to_csv(
            CHECKPOINT_PATH,
            index=False
        )

        elapsed = time.time() - start_time

        verified_count = int(
            verified_df["verified"].sum()
        )

        print(
            f"Verified {completed}/{len(df)} "
            f"| accepted: {verified_count} "
            f"| elapsed: {elapsed/60:.1f} min "
            f"| checkpoint saved"
        )

# ============================================
# FINAL SAVE
# ============================================

verified_df.to_csv(
    FINAL_PATH,
    index=False
)

elapsed = time.time() - start_time

print("\n============================================")
print("MISTRAL VERIFICATION COMPLETE")
print("============================================")

print(
    "Total candidates:",
    len(verified_df)
)

print("\nVerification distribution:")
print(
    verified_df["verified"].value_counts()
)

print("\nVerification percentages:")
print(
    verified_df["verified"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print(
    "\nVerified:",
    int(verified_df["verified"].sum())
)

print(
    "Rejected:",
    int((~verified_df["verified"]).sum())
)

print(
    "Time:",
    round(elapsed / 60, 2),
    "minutes"
)

print(
    "Saved:",
    FINAL_PATH
)

display(
    verified_df.head()
)

CrossEncoder

In [ ]:
# ============================================
# FINAL CROSSENCODER DATASET PREPARATION
# ============================================

import pandas as pd
import numpy as np
import os

from sklearn.model_selection import train_test_split

RANDOM_SEED = 42
SYNTHETIC_SIZE = 3207

# --------------------------------------------
# 1. Load original training data
# --------------------------------------------

TRAIN_PATH = (
    "Task 2/Task 2.1/Train/"
    "train_dataset_sourced.jsonl"
)

ABSTRACT_PATH = (
    "Task 2/Task 2.1/"
    "source_abstracts.jsonl"
)

train = pd.read_json(
    TRAIN_PATH,
    lines=True
)

abstracts = pd.read_json(
    ABSTRACT_PATH,
    lines=True
)

# --------------------------------------------
# 2. Attach source abstracts
# --------------------------------------------

train = train.merge(
    abstracts[["abs_id", "abs_source"]],
    on="abs_id",
    how="left"
)

train["label"] = train["is_spurious"].astype(int)

# --------------------------------------------
# 3. EXACT SAME 90/10 SPLIT
# --------------------------------------------

train_df, valid_df = train_test_split(
    train,
    test_size=0.10,
    random_state=RANDOM_SEED,
    stratify=train["label"]
)

train_df = train_df.reset_index(drop=True)
valid_df = valid_df.reset_index(drop=True)

train_df = train_df[
    ["abs_source", "sentence", "label"]
]

valid_df = valid_df[
    ["abs_source", "sentence", "label"]
]

print("Original data:")
print("  Total:", len(train))
print("  Training:", len(train_df))
print("  Validation:", len(valid_df))

print("\nTraining class distribution:")
print(
    train_df["label"].value_counts()
)

print("\nValidation class distribution:")
print(
    valid_df["label"].value_counts()
)


# ============================================
# 4. LOAD VERIFIED SYNTHETIC DATA
# ============================================

QWEN_PATH = (
    "./synthetic_qwen/"
    "qwen_synthetic_10k_verified.csv"
)

LLAMA_PATH = (
    "./synthetic_llama/"
    "llama_synthetic_10k_verified.csv"
)

MISTRAL_PATH = (
    "./synthetic_mistral/"
    "mistral_synthetic_10k_verified.csv"
)

qwen = pd.read_csv(QWEN_PATH)
llama = pd.read_csv(LLAMA_PATH)
mistral = pd.read_csv(MISTRAL_PATH)

# Keep verified examples only
qwen = qwen[
    qwen["verified"] == True
].copy()

llama = llama[
    llama["verified"] == True
].copy()

mistral = mistral[
    mistral["verified"] == True
].copy()


# ============================================
# 5. FIXED-SIZE SAMPLING
# ============================================

qwen_sample = qwen.sample(
    n=SYNTHETIC_SIZE,
    random_state=RANDOM_SEED
)

llama_sample = llama.sample(
    n=SYNTHETIC_SIZE,
    random_state=RANDOM_SEED
)

mistral_sample = mistral.sample(
    n=SYNTHETIC_SIZE,
    random_state=RANDOM_SEED
)


# ============================================
# 6. STANDARDISE SYNTHETIC COLUMNS
# ============================================

def prepare_synthetic(df):

    return df[
        ["abs_id", "sentence", "label", "source"]
    ].copy()


qwen_sample = prepare_synthetic(
    qwen_sample
)

llama_sample = prepare_synthetic(
    llama_sample
)

mistral_sample = prepare_synthetic(
    mistral_sample
)


# ============================================
# 7. CREATE CROSSENCODER FORMAT
# ============================================

def synthetic_to_training_format(
    synthetic_df
):

    return pd.DataFrame({
        "abs_source": synthetic_df["abs_id"].map(
            abstracts.set_index("abs_id")[
                "abs_source"
            ]
        ),
        "sentence": synthetic_df["sentence"],
        "label": synthetic_df["label"].astype(int)
    })


qwen_train = synthetic_to_training_format(
    qwen_sample
)

llama_train = synthetic_to_training_format(
    llama_sample
)

mistral_train = synthetic_to_training_format(
    mistral_sample
)


# ============================================
# 8. CREATE FOUR CONDITIONS
# ============================================

baseline_train = train_df.copy()

qwen_augmented_train = pd.concat(
    [
        train_df,
        qwen_train
    ],
    ignore_index=True
)

llama_augmented_train = pd.concat(
    [
        train_df,
        llama_train
    ],
    ignore_index=True
)

mistral_augmented_train = pd.concat(
    [
        train_df,
        mistral_train
    ],
    ignore_index=True
)


# ============================================
# 9. SUMMARY
# ============================================

print("\n============================================")
print("FINAL TRAINING DATASETS")
print("============================================")

print(
    "\nBaseline:",
    len(baseline_train)
)

print(
    "Qwen:",
    len(qwen_augmented_train),
    "(+ 3,207)"
)

print(
    "Llama:",
    len(llama_augmented_train),
    "(+ 3,207)"
)

print(
    "Mistral:",
    len(mistral_augmented_train),
    "(+ 3,207)"
)

print(
    "\nValidation:",
    len(valid_df)
)

print("\nClass distributions:")

for name, data in [
    ("Baseline", baseline_train),
    ("Qwen", qwen_augmented_train),
    ("Llama", llama_augmented_train),
    ("Mistral", mistral_augmented_train)
]:

    print(
        f"{name}:",
        data["label"].value_counts().to_dict()
    )

# --------------------------------------------
# Sanity assertion
# --------------------------------------------

assert len(qwen_train) == 3207
assert len(llama_train) == 3207
assert len(mistral_train) == 3207

assert len(valid_df) > 0

print(
    "\nDataset preparation successful!"
)

In [ ]:
# ============================================
# PREPARE CROSSENCODER TRAINING
# ============================================

import os
import torch

from sentence_transformers import CrossEncoder, InputExample
from torch.utils.data import DataLoader
from sentence_transformers.cross_encoder.evaluation import (
    CEBinaryClassificationEvaluator
)

MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"

BATCH_SIZE = 4
EPOCHS = 20
WARMUP_STEPS = 100
MAX_LENGTH = 512
RANDOM_SEED = 42

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

    print(
        "GPU memory allocated:",
        round(
            torch.cuda.memory_allocated() / 1024**3,
            2
        ),
        "GB"
    )

print("\nCrossEncoder configuration:")
print("Model:", MODEL_NAME)
print("Batch size:", BATCH_SIZE)
print("Epochs:", EPOCHS)
print("Warmup steps:", WARMUP_STEPS)
print("Max length:", MAX_LENGTH)
print("Random seed:", RANDOM_SEED)

In [ ]:
# ============================================
# FINAL EXPERIMENT 1 — BASELINE
# ============================================

import os
import torch
import random
import numpy as np

from sentence_transformers import CrossEncoder, InputExample
from sentence_transformers.cross_encoder.evaluation import (
    CEBinaryClassificationEvaluator
)
from torch.utils.data import DataLoader

# --------------------------------------------
# Reproducibility
# --------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# --------------------------------------------
# Paths
# --------------------------------------------

OUTPUT_DIR = "./crossencoder_final/baseline"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

# --------------------------------------------
# Convert baseline dataframe to InputExamples
# --------------------------------------------

baseline_samples = [
    InputExample(
        texts=[abs_source, sentence],
        label=float(label)
    )
    for abs_source, sentence, label
    in zip(
        baseline_train["abs_source"],
        baseline_train["sentence"],
        baseline_train["label"]
    )
]

# --------------------------------------------
# Validation samples
# --------------------------------------------

validation_samples = [
    InputExample(
        texts=[abs_source, sentence],
        label=float(label)
    )
    for abs_source, sentence, label
    in zip(
        valid_df["abs_source"],
        valid_df["sentence"],
        valid_df["label"]
    )
]

print("Baseline training samples:",
      len(baseline_samples))

print("Validation samples:",
      len(validation_samples))

# --------------------------------------------
# DataLoader
# --------------------------------------------

baseline_dataloader = DataLoader(
    baseline_samples,
    shuffle=True,
    batch_size=4
)

# --------------------------------------------
# Evaluator
# --------------------------------------------

baseline_evaluator = (
    CEBinaryClassificationEvaluator
    .from_input_examples(
        validation_samples,
        name="validation"
    )
)

# --------------------------------------------
# CrossEncoder
# --------------------------------------------

baseline_model = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L6-v2",
    num_labels=1,
    max_length=512
)

print("\nBaseline CrossEncoder loaded!")

print(
    "Training on:",
    torch.cuda.get_device_name(0)
    if torch.cuda.is_available()
    else "CPU"
)

# --------------------------------------------
# TRAIN
# --------------------------------------------

print("\n============================================")
print("STARTING BASELINE TRAINING")
print("============================================")

baseline_model.fit(
    train_dataloader=baseline_dataloader,

    evaluator=baseline_evaluator,

    epochs=20,

    warmup_steps=100,

    output_path=OUTPUT_DIR,

    save_best_model=True
)

print("\n============================================")
print("BASELINE TRAINING COMPLETE")
print("============================================")

print(
    "Best model saved to:",
    OUTPUT_DIR
)

In [ ]:
# ============================================
# BASELINE — VALIDATION PERFORMANCE
# ============================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)

print("Using trained baseline_model from memory.")

# --------------------------------------------
# Validation pairs
# --------------------------------------------

valid_pairs = list(
    zip(
        valid_df["abs_source"],
        valid_df["sentence"]
    )
)

y_true = (
    valid_df["label"]
    .astype(int)
    .values
)

# --------------------------------------------
# Predictions
# --------------------------------------------

print("\nGenerating validation predictions...")

y_scores = baseline_model.predict(
    valid_pairs,
    show_progress_bar=True
)

y_scores = np.asarray(
    y_scores
).reshape(-1)

# Binary predictions
y_pred = (
    y_scores >= 0.5
).astype(int)

# --------------------------------------------
# Metrics
# --------------------------------------------

accuracy = accuracy_score(
    y_true,
    y_pred
)

precision, recall, f1, _ = (
    precision_recall_fscore_support(
        y_true,
        y_pred,
        average="binary",
        zero_division=0
    )
)

auroc = roc_auc_score(
    y_true,
    y_scores
)

auprc = average_precision_score(
    y_true,
    y_scores
)

# --------------------------------------------
# Save results
# --------------------------------------------

baseline_results = {
    "model": "Baseline",
    "accuracy": accuracy,
    "precision": precision,
    "recall": recall,
    "f1": f1,
    "auroc": auroc,
    "auprc": auprc
}

# --------------------------------------------
# Display
# --------------------------------------------

print("\n============================================")
print("BASELINE VALIDATION RESULTS")
print("============================================")

for metric, value in baseline_results.items():

    if metric != "model":

        print(
            f"{metric.upper():10s}: "
            f"{value:.4f}"
        )

print("\nClassification report:")
print(
    classification_report(
        y_true,
        y_pred,
        target_names=[
            "Not-Spurious",
            "Spurious"
        ],
        zero_division=0
    )
)

print("\nConfusion matrix:")
print(
    confusion_matrix(
        y_true,
        y_pred
    )
)

# --------------------------------------------
# Save metrics
# --------------------------------------------

os.makedirs(
    "./crossencoder_final/results",
    exist_ok=True
)

pd.DataFrame(
    [baseline_results]
).to_csv(
    "./crossencoder_final/results/"
    "baseline_validation_metrics.csv",
    index=False
)

print(
    "\nResults saved to:"
)

print(
    "./crossencoder_final/results/"
    "baseline_validation_metrics.csv"
)

In [ ]:
# ============================================
# FINAL EXPERIMENT 2 — QWEN AUGMENTED
# ============================================

import os
import random
import numpy as np
import torch

from sentence_transformers import CrossEncoder, InputExample
from sentence_transformers.cross_encoder.evaluation import (
    CEBinaryClassificationEvaluator
)
from torch.utils.data import DataLoader

# --------------------------------------------
# Reproducibility
# --------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# --------------------------------------------
# Output directory
# --------------------------------------------

OUTPUT_DIR = "./crossencoder_final/qwen"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

# --------------------------------------------
# Create training samples
# --------------------------------------------

qwen_samples = [
    InputExample(
        texts=[abs_source, sentence],
        label=float(label)
    )
    for abs_source, sentence, label
    in zip(
        qwen_augmented_train["abs_source"],
        qwen_augmented_train["sentence"],
        qwen_augmented_train["label"]
    )
]

# --------------------------------------------
# Validation samples
# --------------------------------------------

validation_samples = [
    InputExample(
        texts=[abs_source, sentence],
        label=float(label)
    )
    for abs_source, sentence, label
    in zip(
        valid_df["abs_source"],
        valid_df["sentence"],
        valid_df["label"]
    )
]

print("Qwen training samples:",
      len(qwen_samples))

print("Validation samples:",
      len(validation_samples))

# --------------------------------------------
# DataLoader
# --------------------------------------------

qwen_dataloader = DataLoader(
    qwen_samples,
    shuffle=True,
    batch_size=4
)

# --------------------------------------------
# Evaluator
# --------------------------------------------

qwen_evaluator = (
    CEBinaryClassificationEvaluator
    .from_input_examples(
        validation_samples,
        name="validation"
    )
)

# --------------------------------------------
# Model
# --------------------------------------------

qwen_crossencoder = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L6-v2",
    num_labels=1,
    max_length=512
)

print("\nQwen CrossEncoder loaded!")

# --------------------------------------------
# TRAIN
# --------------------------------------------

print("\n============================================")
print("STARTING QWEN-AUGMENTED TRAINING")
print("============================================")

qwen_crossencoder.fit(
    train_dataloader=qwen_dataloader,

    evaluator=qwen_evaluator,

    epochs=20,

    warmup_steps=100,

    output_path=OUTPUT_DIR,

    save_best_model=True
)

print("\n============================================")
print("QWEN TRAINING COMPLETE")
print("============================================")

print(
    "Best model saved to:",
    OUTPUT_DIR
)

In [ ]:
# ============================================
# QWEN — VALIDATION PERFORMANCE
# ============================================

import numpy as np
import pandas as pd
import os

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)

print("Using trained qwen_crossencoder from memory.")

valid_pairs = list(
    zip(
        valid_df["abs_source"],
        valid_df["sentence"]
    )
)

y_true = (
    valid_df["label"]
    .astype(int)
    .values
)

print("\nGenerating Qwen validation predictions...")

y_scores = qwen_crossencoder.predict(
    valid_pairs,
    show_progress_bar=True
)

y_scores = np.asarray(
    y_scores
).reshape(-1)

y_pred = (
    y_scores >= 0.5
).astype(int)

# --------------------------------------------
# Metrics
# --------------------------------------------

accuracy = accuracy_score(
    y_true,
    y_pred
)

precision, recall, f1, _ = (
    precision_recall_fscore_support(
        y_true,
        y_pred,
        average="binary",
        zero_division=0
    )
)

auroc = roc_auc_score(
    y_true,
    y_scores
)

auprc = average_precision_score(
    y_true,
    y_scores
)

qwen_results = {
    "model": "Qwen",
    "accuracy": accuracy,
    "precision": precision,
    "recall": recall,
    "f1": f1,
    "auroc": auroc,
    "auprc": auprc
}

print("\n============================================")
print("QWEN VALIDATION RESULTS")
print("============================================")

for metric, value in qwen_results.items():

    if metric != "model":
        print(
            f"{metric.upper():10s}: "
            f"{value:.4f}"
        )

print("\nClassification report:")
print(
    classification_report(
        y_true,
        y_pred,
        target_names=[
            "Not-Spurious",
            "Spurious"
        ],
        zero_division=0
    )
)

print("\nConfusion matrix:")
print(
    confusion_matrix(
        y_true,
        y_pred
    )
)

# --------------------------------------------
# Save
# --------------------------------------------

os.makedirs(
    "./crossencoder_final/results",
    exist_ok=True
)

pd.DataFrame(
    [qwen_results]
).to_csv(
    "./crossencoder_final/results/"
    "qwen_validation_metrics.csv",
    index=False
)

print(
    "\nResults saved to:"
)

print(
    "./crossencoder_final/results/"
    "qwen_validation_metrics.csv"
)

In [ ]:
# ============================================
# FINAL EXPERIMENT 3 — LLAMA AUGMENTED
# ============================================

import os
import random
import numpy as np
import torch

from sentence_transformers import CrossEncoder, InputExample
from sentence_transformers.cross_encoder.evaluation import (
    CEBinaryClassificationEvaluator
)
from torch.utils.data import DataLoader

# --------------------------------------------
# Reproducibility
# --------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# --------------------------------------------
# Configuration
# --------------------------------------------

MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L6-v2"

BATCH_SIZE = 4
EPOCHS = 20
WARMUP_STEPS = 100
MAX_LENGTH = 512

OUTPUT_DIR = "./crossencoder_final/llama"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

# --------------------------------------------
# Llama training samples
# --------------------------------------------

llama_samples = [
    InputExample(
        texts=[abs_source, sentence],
        label=float(label)
    )
    for abs_source, sentence, label
    in zip(
        llama_augmented_train["abs_source"],
        llama_augmented_train["sentence"],
        llama_augmented_train["label"]
    )
]

# --------------------------------------------
# Same validation set
# --------------------------------------------

validation_samples = [
    InputExample(
        texts=[abs_source, sentence],
        label=float(label)
    )
    for abs_source, sentence, label
    in zip(
        valid_df["abs_source"],
        valid_df["sentence"],
        valid_df["label"]
    )
]

print("Llama training samples:",
      len(llama_samples))

print("Validation samples:",
      len(validation_samples))

# --------------------------------------------
# DataLoader
# --------------------------------------------

llama_dataloader = DataLoader(
    llama_samples,
    shuffle=True,
    batch_size=BATCH_SIZE
)

# --------------------------------------------
# Evaluator
# --------------------------------------------

llama_evaluator = (
    CEBinaryClassificationEvaluator
    .from_input_examples(
        validation_samples,
        name="validation"
    )
)

# --------------------------------------------
# Load CrossEncoder
# --------------------------------------------

llama_crossencoder = CrossEncoder(
    MODEL_NAME,
    num_labels=1,
    max_length=MAX_LENGTH
)

print("\nLlama CrossEncoder loaded!")

print(
    "Training on:",
    torch.cuda.get_device_name(0)
    if torch.cuda.is_available()
    else "CPU"
)

# --------------------------------------------
# TRAIN
# --------------------------------------------

print("\n============================================")
print("STARTING LLAMA-AUGMENTED TRAINING")
print("============================================")

llama_crossencoder.fit(
    train_dataloader=llama_dataloader,

    evaluator=llama_evaluator,

    epochs=EPOCHS,

    warmup_steps=WARMUP_STEPS,

    output_path=OUTPUT_DIR,

    save_best_model=True
)

print("\n============================================")
print("LLAMA TRAINING COMPLETE")
print("============================================")

print(
    "Best model saved to:",
    OUTPUT_DIR
)

In [ ]:
# ============================================
# LLAMA — VALIDATION PERFORMANCE
# ============================================

import numpy as np
import pandas as pd
import os

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)

print("Using trained llama_crossencoder from memory.")

# --------------------------------------------
# Validation pairs
# --------------------------------------------

valid_pairs = list(
    zip(
        valid_df["abs_source"],
        valid_df["sentence"]
    )
)

y_true = (
    valid_df["label"]
    .astype(int)
    .values
)

# --------------------------------------------
# Predictions
# --------------------------------------------

print("\nGenerating Llama validation predictions...")

y_scores = llama_crossencoder.predict(
    valid_pairs,
    show_progress_bar=True
)

y_scores = np.asarray(
    y_scores
).reshape(-1)

y_pred = (
    y_scores >= 0.5
).astype(int)

# --------------------------------------------
# Metrics
# --------------------------------------------

accuracy = accuracy_score(
    y_true,
    y_pred
)

precision, recall, f1, _ = (
    precision_recall_fscore_support(
        y_true,
        y_pred,
        average="binary",
        zero_division=0
    )
)

auroc = roc_auc_score(
    y_true,
    y_scores
)

auprc = average_precision_score(
    y_true,
    y_scores
)

llama_results = {
    "model": "Llama",
    "accuracy": accuracy,
    "precision": precision,
    "recall": recall,
    "f1": f1,
    "auroc": auroc,
    "auprc": auprc
}

# --------------------------------------------
# Results
# --------------------------------------------

print("\n============================================")
print("LLAMA VALIDATION RESULTS")
print("============================================")

for metric, value in llama_results.items():

    if metric != "model":
        print(
            f"{metric.upper():10s}: "
            f"{value:.4f}"
        )

print("\nClassification report:")

print(
    classification_report(
        y_true,
        y_pred,
        target_names=[
            "Not-Spurious",
            "Spurious"
        ],
        zero_division=0
    )
)

print("\nConfusion matrix:")

print(
    confusion_matrix(
        y_true,
        y_pred
    )
)

# --------------------------------------------
# Save
# --------------------------------------------

os.makedirs(
    "./crossencoder_final/results",
    exist_ok=True
)

pd.DataFrame(
    [llama_results]
).to_csv(
    "./crossencoder_final/results/"
    "llama_validation_metrics.csv",
    index=False
)

print(
    "\nResults saved to:"
)

print(
    "./crossencoder_final/results/"
    "llama_validation_metrics.csv"
)

In [ ]:
# ============================================
# FINAL EXPERIMENT 4 — MISTRAL AUGMENTED
# ============================================

import os
import random
import numpy as np
import torch

from sentence_transformers import CrossEncoder, InputExample
from sentence_transformers.cross_encoder.evaluation import (
    CEBinaryClassificationEvaluator
)
from torch.utils.data import DataLoader

# --------------------------------------------
# Reproducibility
# --------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# --------------------------------------------
# Configuration
# --------------------------------------------

MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L6-v2"

BATCH_SIZE = 4
EPOCHS = 20
WARMUP_STEPS = 100
MAX_LENGTH = 512

OUTPUT_DIR = "./crossencoder_final/mistral"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

# --------------------------------------------
# Mistral training samples
# --------------------------------------------

mistral_samples = [
    InputExample(
        texts=[abs_source, sentence],
        label=float(label)
    )
    for abs_source, sentence, label
    in zip(
        mistral_augmented_train["abs_source"],
        mistral_augmented_train["sentence"],
        mistral_augmented_train["label"]
    )
]

# --------------------------------------------
# Same validation set
# --------------------------------------------

validation_samples = [
    InputExample(
        texts=[abs_source, sentence],
        label=float(label)
    )
    for abs_source, sentence, label
    in zip(
        valid_df["abs_source"],
        valid_df["sentence"],
        valid_df["label"]
    )
]

print("Mistral training samples:",
      len(mistral_samples))

print("Validation samples:",
      len(validation_samples))

# --------------------------------------------
# DataLoader
# --------------------------------------------

mistral_dataloader = DataLoader(
    mistral_samples,
    shuffle=True,
    batch_size=BATCH_SIZE
)

# --------------------------------------------
# Evaluator
# --------------------------------------------

mistral_evaluator = (
    CEBinaryClassificationEvaluator
    .from_input_examples(
        validation_samples,
        name="validation"
    )
)

# --------------------------------------------
# CrossEncoder
# --------------------------------------------

mistral_crossencoder = CrossEncoder(
    MODEL_NAME,
    num_labels=1,
    max_length=MAX_LENGTH
)

print("\nMistral CrossEncoder loaded!")

print(
    "Training on:",
    torch.cuda.get_device_name(0)
    if torch.cuda.is_available()
    else "CPU"
)

# --------------------------------------------
# TRAIN
# --------------------------------------------

print("\n============================================")
print("STARTING MISTRAL-AUGMENTED TRAINING")
print("============================================")

mistral_crossencoder.fit(
    train_dataloader=mistral_dataloader,

    evaluator=mistral_evaluator,

    epochs=EPOCHS,

    warmup_steps=WARMUP_STEPS,

    output_path=OUTPUT_DIR,

    save_best_model=True
)

print("\n============================================")
print("MISTRAL TRAINING COMPLETE")
print("============================================")

print(
    "Best model saved to:",
    OUTPUT_DIR
)

In [ ]:
# ============================================
# MISTRAL — VALIDATION PERFORMANCE
# ============================================

import numpy as np
import pandas as pd
import os

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)

print("Using trained mistral_crossencoder from memory.")

# --------------------------------------------
# Validation pairs
# --------------------------------------------

valid_pairs = list(
    zip(
        valid_df["abs_source"],
        valid_df["sentence"]
    )
)

y_true = (
    valid_df["label"]
    .astype(int)
    .values
)

# --------------------------------------------
# Predictions
# --------------------------------------------

print("\nGenerating Mistral validation predictions...")

y_scores = mistral_crossencoder.predict(
    valid_pairs,
    show_progress_bar=True
)

y_scores = np.asarray(
    y_scores
).reshape(-1)

y_pred = (
    y_scores >= 0.5
).astype(int)

# --------------------------------------------
# Metrics
# --------------------------------------------

accuracy = accuracy_score(
    y_true,
    y_pred
)

precision, recall, f1, _ = (
    precision_recall_fscore_support(
        y_true,
        y_pred,
        average="binary",
        zero_division=0
    )
)

auroc = roc_auc_score(
    y_true,
    y_scores
)

auprc = average_precision_score(
    y_true,
    y_scores
)

mistral_results = {
    "model": "Mistral",
    "accuracy": accuracy,
    "precision": precision,
    "recall": recall,
    "f1": f1,
    "auroc": auroc,
    "auprc": auprc
}

# --------------------------------------------
# Display
# --------------------------------------------

print("\n============================================")
print("MISTRAL VALIDATION RESULTS")
print("============================================")

for metric, value in mistral_results.items():

    if metric != "model":

        print(
            f"{metric.upper():10s}: "
            f"{value:.4f}"
        )

print("\nClassification report:")

print(
    classification_report(
        y_true,
        y_pred,
        target_names=[
            "Not-Spurious",
            "Spurious"
        ],
        zero_division=0
    )
)

print("\nConfusion matrix:")

print(
    confusion_matrix(
        y_true,
        y_pred
    )
)

# --------------------------------------------
# Save
# --------------------------------------------

os.makedirs(
    "./crossencoder_final/results",
    exist_ok=True
)

pd.DataFrame(
    [mistral_results]
).to_csv(
    "./crossencoder_final/results/"
    "mistral_validation_metrics.csv",
    index=False
)

print(
    "\nResults saved to:"
)

print(
    "./crossencoder_final/results/"
    "mistral_validation_metrics.csv"
)

In [ ]:
# ============================================
# FINAL EXPERIMENT — COMBINE ALL RESULTS
# ============================================

import pandas as pd
import os

RESULTS_DIR = "./crossencoder_final/results"

files = {
    "Baseline": "baseline_validation_metrics.csv",
    "Qwen": "qwen_validation_metrics.csv",
    "Llama": "llama_validation_metrics.csv",
    "Mistral": "mistral_validation_metrics.csv"
}

results = []

for model_name, filename in files.items():

    path = os.path.join(
        RESULTS_DIR,
        filename
    )

    df = pd.read_csv(path)

    df["model"] = model_name

    results.append(df)

final_results = pd.concat(
    results,
    ignore_index=True
)

final_results = final_results[
    [
        "model",
        "accuracy",
        "precision",
        "recall",
        "f1",
        "auroc",
        "auprc"
    ]
]

print("\n============================================")
print("FINAL CROSSENCODER RESULTS")
print("============================================")

display(
    final_results.style.format({
        "accuracy": "{:.4f}",
        "precision": "{:.4f}",
        "recall": "{:.4f}",
        "f1": "{:.4f}",
        "auroc": "{:.4f}",
        "auprc": "{:.4f}"
    })
)

FINAL_RESULTS_PATH = (
    "./crossencoder_final/"
    "final_experiment_results.csv"
)

final_results.to_csv(
    FINAL_RESULTS_PATH,
    index=False
)

print(
    "\nSaved:",
    FINAL_RESULTS_PATH
)

In [ ]:
# ============================================
# STEP 2 — CLASS-WISE FINAL COMPARISON
# ============================================

import numpy as np
import pandas as pd
import os

from sklearn.metrics import (
    classification_report,
    confusion_matrix
)

models = {
    "Baseline": baseline_model,
    "Qwen": qwen_crossencoder,
    "Llama": llama_crossencoder,
    "Mistral": mistral_crossencoder
}

# --------------------------------------------
# Common validation set
# --------------------------------------------

valid_pairs = list(
    zip(
        valid_df["abs_source"],
        valid_df["sentence"]
    )
)

y_true = (
    valid_df["label"]
    .astype(int)
    .values
)

classwise_results = []
confusion_matrices = {}

# --------------------------------------------
# Evaluate every model
# --------------------------------------------

for model_name, model in models.items():

    print(
        f"\nEvaluating {model_name}..."
    )

    scores = model.predict(
        valid_pairs,
        show_progress_bar=False
    )

    scores = np.asarray(
        scores
    ).reshape(-1)

    predictions = (
        scores >= 0.5
    ).astype(int)

    report = classification_report(
        y_true,
        predictions,
        output_dict=True,
        zero_division=0
    )

    # ----------------------------------------
    # Class 0 — Not-Spurious
    # ----------------------------------------

    classwise_results.append({
        "model": model_name,
        "class": "Not-Spurious",
        "precision": report["0"]["precision"],
        "recall": report["0"]["recall"],
        "f1": report["0"]["f1-score"],
        "support": report["0"]["support"]
    })

    # ----------------------------------------
    # Class 1 — Spurious
    # ----------------------------------------

    classwise_results.append({
        "model": model_name,
        "class": "Spurious",
        "precision": report["1"]["precision"],
        "recall": report["1"]["recall"],
        "f1": report["1"]["f1-score"],
        "support": report["1"]["support"]
    })

    # ----------------------------------------
    # Macro / weighted averages
    # ----------------------------------------

    classwise_results.append({
        "model": model_name,
        "class": "Macro Avg",
        "precision": report["macro avg"]["precision"],
        "recall": report["macro avg"]["recall"],
        "f1": report["macro avg"]["f1-score"],
        "support": len(y_true)
    })

    classwise_results.append({
        "model": model_name,
        "class": "Weighted Avg",
        "precision": report["weighted avg"]["precision"],
        "recall": report["weighted avg"]["recall"],
        "f1": report["weighted avg"]["f1-score"],
        "support": len(y_true)
    })

    # ----------------------------------------
    # Confusion matrix
    # ----------------------------------------

    confusion_matrices[model_name] = (
        confusion_matrix(
            y_true,
            predictions
        )
    )

# ============================================
# CLASS-WISE TABLE
# ============================================

classwise_df = pd.DataFrame(
    classwise_results
)

print("\n============================================")
print("CLASS-WISE PERFORMANCE")
print("============================================")

display(
    classwise_df.style.format({
        "precision": "{:.4f}",
        "recall": "{:.4f}",
        "f1": "{:.4f}"
    })
)

# ============================================
# CONFUSION MATRICES
# ============================================

print("\n============================================")
print("CONFUSION MATRICES")
print("============================================")

for model_name, cm in confusion_matrices.items():

    print(f"\n{model_name}")
    print(
        "                 Predicted"
    )
    print(
        "               Not-Spur  Spur"
    )
    print(
        f"Actual Not-Spur   {cm[0,0]:4d}   {cm[0,1]:4d}"
    )
    print(
        f"       Spurious   {cm[1,0]:4d}   {cm[1,1]:4d}"
    )

# ============================================
# SAVE
# ============================================

CLASSWISE_PATH = (
    "./crossencoder_final/"
    "classwise_results.csv"
)

classwise_df.to_csv(
    CLASSWISE_PATH,
    index=False
)

print(
    "\nSaved:",
    CLASSWISE_PATH
)

In [ ]:
# ============================================================
# LOAD BASELINE CHECKPOINT FOR EVALUATION
# ============================================================

import torch
from transformers import AutoConfig, AutoTokenizer, AutoModelForSequenceClassification

BASE_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
BASELINE_CHECKPOINT = "./crossencoder_results/checkpoint-60820"

print("=" * 70)
print("LOADING BASELINE MODEL")
print("=" * 70)

# Load tokenizer from the original pretrained model
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL
)

# Load the checkpoint configuration
config = AutoConfig.from_pretrained(
    BASELINE_CHECKPOINT
)

# Load the trained checkpoint weights
hf_model = AutoModelForSequenceClassification.from_pretrained(
    BASELINE_CHECKPOINT,
    config=config,
    use_safetensors=True
)

# Put model on GPU
hf_model = hf_model.to("cuda")
hf_model.eval()

print("✅ Baseline tokenizer loaded")
print("✅ Baseline checkpoint loaded")
print("Model type:", type(hf_model).__name__)
print("Device:", next(hf_model.parameters()).device)

In [ ]:
# ============================================================
# FINAL CORRECTED CROSS-LLM SYNTHETIC VALIDATION
# ============================================================

import os
import numpy as np
import pandas as pd
import torch

print("=" * 70)
print("FINAL CROSS-LLM SYNTHETIC VALIDATION")
print("=" * 70)

results = []

# ============================================================
# 1. CORRECT BASELINE PREDICTOR
# ============================================================

def baseline_predict_labels(pairs, batch_size=32):

    all_logits = []

    for start in range(0, len(pairs), batch_size):

        batch = pairs[start:start + batch_size]

        texts_a = [x[0] for x in batch]
        texts_b = [x[1] for x in batch]

        encoded = tokenizer(
            texts_a,
            texts_b,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt"
        )

        encoded = {
            k: v.to("cuda")
            for k, v in encoded.items()
        }

        with torch.no_grad():

            outputs = hf_model(**encoded)

            logits = outputs.logits

        all_logits.append(
            logits.detach().cpu().numpy()
        )

    logits = np.concatenate(
        all_logits,
        axis=0
    )

    # 2-class prediction
    predicted_labels = np.argmax(
        logits,
        axis=1
    )

    return predicted_labels, logits


# ============================================================
# 2. EVALUATE ALL THREE SYNTHETIC DATASETS
# ============================================================

for generator, df in synthetic_data.items():

    print("\n" + "=" * 70)
    print(f"{generator.upper()} SYNTHETIC DATA")
    print("=" * 70)

    pairs = list(
        zip(
            df["abs_source"].astype(str),
            df["sentence"].astype(str)
        )
    )

    # --------------------------------------------------------
    # BASELINE
    # --------------------------------------------------------

    print(f"\nBaseline → {generator}")

    predicted_labels, logits = baseline_predict_labels(
        pairs
    )

    not_spurious = int(
        np.sum(predicted_labels == 0)
    )

    spurious = int(
        np.sum(predicted_labels == 1)
    )

    total = len(df)

    print(
        f"Not-Spurious: "
        f"{not_spurious}/{total} "
        f"({not_spurious / total:.2%})"
    )

    print(
        f"Spurious: "
        f"{spurious}/{total} "
        f"({spurious / total:.2%})"
    )

    results.append({
        "generator": generator,
        "evaluator": "Baseline",
        "samples": total,
        "predicted_not_spurious": not_spurious,
        "predicted_spurious": spurious,
        "not_spurious_rate": not_spurious / total,
        "spurious_rate": spurious / total,
        "mean_score": float(
            logits[:, 1].mean()
        ),
        "median_score": float(
            np.median(logits[:, 1])
        )
    })

    # --------------------------------------------------------
    # QWEN / LLAMA / MISTRAL
    # --------------------------------------------------------

    for evaluator_name in [
        "Qwen",
        "Llama",
        "Mistral"
    ]:

        print(
            f"\n{evaluator_name} → {generator}"
        )

        model = models[evaluator_name]

        scores = model.predict(
            pairs,
            show_progress_bar=True
        )

        scores = np.asarray(
            scores
        ).reshape(-1)

        predicted_labels = (
            scores >= 0
        ).astype(int)

        not_spurious = int(
            np.sum(predicted_labels == 0)
        )

        spurious = int(
            np.sum(predicted_labels == 1)
        )

        print(
            f"Not-Spurious: "
            f"{not_spurious}/{total} "
            f"({not_spurious / total:.2%})"
        )

        print(
            f"Spurious: "
            f"{spurious}/{total} "
            f"({spurious / total:.2%})"
        )

        results.append({
            "generator": generator,
            "evaluator": evaluator_name,
            "samples": total,
            "predicted_not_spurious": not_spurious,
            "predicted_spurious": spurious,
            "not_spurious_rate": not_spurious / total,
            "spurious_rate": spurious / total,
            "mean_score": float(
                scores.mean()
            ),
            "median_score": float(
                np.median(scores)
            )
        })


# ============================================================
# 3. CREATE FINAL RESULTS DATAFRAME
# ============================================================

cross_llm_results = pd.DataFrame(
    results
)

# Safety check
assert all(
    cross_llm_results["predicted_not_spurious"]
    +
    cross_llm_results["predicted_spurious"]
    ==
    cross_llm_results["samples"]
)

# ============================================================
# 4. SAVE
# ============================================================

output_dir = "./crossencoder_final/analysis"

os.makedirs(
    output_dir,
    exist_ok=True
)

output_path = os.path.join(
    output_dir,
    "cross_llm_synthetic_validation.csv"
)

cross_llm_results.to_csv(
    output_path,
    index=False
)

# ============================================================
# 5. FINAL TABLE
# ============================================================

print("\n" + "=" * 70)
print("FINAL CROSS-LLM SYNTHETIC VALIDATION")
print("=" * 70)

display(
    cross_llm_results[
        [
            "generator",
            "evaluator",
            "samples",
            "predicted_not_spurious",
            "predicted_spurious",
            "not_spurious_rate",
            "spurious_rate",
            "mean_score",
            "median_score"
        ]
    ]
)

print("\nSaved:")
print(output_path)